In [1]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists("Scaling-book"):
        !git clone https://github.com/arjuns238/Scaling-book.git
    %cd Scaling-book/Addition_Transformer
    !pip install -q flax optax

Cloning into 'Scaling-book'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 95 (delta 55), reused 59 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 218.47 KiB | 8.74 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/Scaling-book/Addition_Transformer


In [3]:
import jax
jax.devices()

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]

In [4]:
# # chose configs - ~10M params
from model import *
from data import build_dataset, generate_split, Dataloader
import numpy as np
from config import Config
base_cfg = Config(
    d_model=384,
    ffw_multiplier=8/3,
    num_layers=6,
    query_heads=6,
    kv_heads=6,
    key_dim=64,
    vocab_size=16,
    batch_size = 256,
    dtype=jnp.bfloat16,
    lr = 1e-3,
    num_epochs = 15,
)

In [4]:
from utils import *

# Precompute (d_model, num_layers, N) once so we can pick sizes by N.
def size_table(base, SIZE_LADDER):
    table = []
    for dm, nl in SIZE_LADDER:
        w = Weights.init(make_cfg(base, dm, nl), jax.random.key(0))
        table.append((dm, nl, count_params(w)))
    return table  # list of (d_model, num_layers, N)

The smallest model is N=100,416. 

With N_min = 100416 and SUP_PER_EX ≈ 4 ( supervised tokens per example —, it's roughly the answer digits + EOS, so ~4 for the 2-and-3-digit-dominated draw).

The hard ceiling is distinct non-fundamental pairs: ~999,900. Fundamental pairs are defined as the cases with 1-digit addition as they are the fundamentals for all other cases. These are excluded from loss calculation.

To stay single-epoch (all fresh data, clean Chinchilla regime), the largest feasible budget for the smallest model:

C_max <= 6 * N_min * SUP_PER_EX * POOL_EXAMPLES
      =  6 * 100416 * 4 * 999900
      ≈  2,409,743,001,600
      ≈  2.4e12

In [5]:
from data import create_example
def build_full_pool(max_seq_len, max_n=1000, seed=0, exclude_both_1digit=True):
    pairs = []
    for a in range(max_n):
        for b in range(max_n):
            if exclude_both_1digit and a < 10 and b < 10:
                continue
            pairs.append((a, b))
    rng = np.random.default_rng(seed)
    rng.shuffle(pairs)  # so any prefix is a representative sample, not a-major

    n = len(pairs)
    tokens = np.empty((n, max_seq_len), dtype=np.int32)
    masks  = np.empty((n, max_seq_len), dtype=np.int32)
    for i, (a, b) in enumerate(pairs):
        t, m = create_example(a, b, max_seq_len)   # your fn
        tokens[i] = t
        masks[i]  = m
    return tokens, masks

In [9]:
def budget_ceiling(N_min, sup_per_ex, n_unique_train):
    """Largest C at which the smallest model still trains single-epoch."""
    D_max_tokens = n_unique_train * sup_per_ex
    return 6 * N_min * D_max_tokens

def examples_for(C, N, sup_per_ex):
    return int(round(C / (6 * N * sup_per_ex)))

In [10]:
# ----------------------------------------------------------------------
# 5. Corner check: 4 extremes -> go/no-go before the full sweep.
# ----------------------------------------------------------------------
def corner_check(base_cfg, tbl, budgets, train_tok, train_mask,
                 val_tok, val_mask, sup_per_ex, batch_size):
    n_train = len(train_tok)
    smallest, largest = tbl[0], tbl[-1]
    corners = [(budgets[0], smallest), (budgets[0], largest),
               (budgets[-1], smallest), (budgets[-1], largest)]
    print(corners)
    print(f"{'C':>10} {'N':>11} {'examples':>10} {'feasible':>9}  val_loss")
    results = []
    for C, (dm, nl, N) in corners:
        ne = examples_for(C, N, sup_per_ex)
        print("ne",ne) 
        feasible = ne <= n_train
        if feasible:
            vl = train_isoflop_point(
                make_cfg(base_cfg, dm, nl), train_tok, train_mask,
                val_tok, val_mask, ne, batch_size, base_cfg.lr)
        else:
            vl = float("nan")
        results.append((C, N, ne, feasible, vl))
        print(f"{C:>10.1e} {N:>11,} {ne:>10,} {str(feasible):>9}  {vl:.4f}")

    losses = [r[4] for r in results if r[3]]
    if any(not r[3] for r in results):
        print("\n[!] Some corners INFEASIBLE (need repeats). Lower C_max.")
    elif len(losses) >= 2 and (max(losses) - min(losses)) < 0.05:
        print("\n[!] Corner losses nearly identical -> valleys likely flat "
              "(saturated or starved). Adjust budget range before full sweep.")
    else:
        print(f"\n[ok] Corners span {max(losses)-min(losses):.3f} loss -> "
              "grid has dynamic range. Proceed to full sweep.")
    return results

In [11]:
POOL_TOK, POOL_MASK = build_full_pool(cfg.max_seq_len, seed=0)
SUP_PER_EX = float(POOL_MASK.sum(axis=1).mean())
print(f"Average tokens per example: {SUP_PER_EX}")

Average tokens per example: 4.494599459945994


In [12]:
BATCH_SIZE = 256
VAL_SIZE = 20_000

SIZE_LADDER = [(64, 2), (96, 3), (128, 3), (192, 4), (256, 5), (384, 6)]
POOL_TOK, POOL_MASK = build_full_pool(cfg.max_seq_len, seed=0)
VAL_TOK,   VAL_MASK   = POOL_TOK[-VAL_SIZE:], POOL_MASK[-VAL_SIZE:]
TRAIN_TOK, TRAIN_MASK = POOL_TOK[:-VAL_SIZE], POOL_MASK[:-VAL_SIZE]

SUP_PER_EX = float(POOL_MASK.sum(axis=1).mean()) # average output tokens per example ≈ 4

# --- size ladder with real N ---
TBL = size_table(cfg, SIZE_LADDER)
N_MIN = min(N for *_, N in TBL)

# --- budget ceiling + working range ---
C_MAX = budget_ceiling(N_MIN, SUP_PER_EX, len(TRAIN_TOK))
C_MIN  = C_MAX / 2e1           # 3 decades; corner check will validate
BUDGETS = np.logspace(np.log10(C_MIN), np.log10(C_MAX), 5)
print(f"C_max={C_MAX:.2e}  C_min={C_MIN:.2e}")

# --- calibrate before committing ---
results = corner_check(cfg, TBL, BUDGETS, TRAIN_TOK, TRAIN_MASK,
                VAL_TOK, VAL_MASK, SUP_PER_EX, BATCH_SIZE)

C_max=2.65e+12  C_min=1.33e+11
[(np.float64(132677391723.78036), (64, 2, 100416)), (np.float64(132677391723.78036), (384, 6, 10634112)), (np.float64(2653547834475.6074), (64, 2, 100416)), (np.float64(2653547834475.6074), (384, 6, 10634112))]
         C           N   examples  feasible  val_loss
ne 48995
n_examples 48995
total steps 191
   1.3e+11     100,416     48,995      True  1.5989
ne 463
n_examples 463
total steps 1


ValueError: The cosine_decay_schedule requires positive decay_steps, got decay_steps=-49.

In [ ]:
# ----------------------------------------------------------------------
# 6. Full sweep: every (budget, size) point.
# ----------------------------------------------------------------------
import itertools
import pandas as pd

def run_sweep(base_cfg, tbl, budgets, train_tok, train_mask,
              val_tok, val_mask, sup_per_ex, batch_size):
    n_train = len(train_tok)
    rows = []
    for C, (dm, nl, N) in itertools.product(budgets, tbl):
        ne = examples_for(C, N, sup_per_ex)
        if ne > n_train or ne < batch_size:
            print(f"skip C={C:.1e} N={N:,} (ne={ne:,} out of range)")
            continue
        vl = train_isoflop_point(
            make_cfg(base_cfg, dm, nl), train_tok, train_mask,
            val_tok, val_mask, ne, batch_size, base_cfg.lr)
        D = ne * sup_per_ex
        rows.append({"C": C, "d_model": dm, "num_layers": nl, "N": N,
                     "n_examples": ne, "D": D, "val_loss": vl})
        print(f"C={C:.1e} N={N:>11,} ne={ne:>8,} -> val {vl:.4f}")
    return pd.DataFrame(rows)

# PART 2

In [5]:
import jax
import jax.numpy as jnp
import numpy as np
import optax
from scipy.optimize import least_squares
from config import Config

BATCH_SIZE     = 256
SEED           = 0
VAL_SUP_TOKENS = 100_000      # fixed val set, counted in supervised tokens
MAX_LEN        = 14
MAX_N          = 1000         # operands 0..999

# model ladder: (d_model, num_layers) — the only two things varied
MODEL_LADDER = [(64, 2), (96, 3), (128, 3), (192, 4), (256, 5), (384, 6)]

# per-model-size peak LR, held fixed across D
PEAK_LR = {
    (64, 2):  1.5e-3,
    (96, 3):  1.2e-3,
    (128, 3): 1.0e-3,
    (192, 4): 8e-4,
    (256, 5): 6e-4,
    (384, 6): 5e-4,
}

# data ladder in SUPERVISED tokens, log-spaced out to the pool ceiling
# (~999,900 non-fundamental examples * ~4.6 sup-tok/ex, minus the val carve)
DATA_LADDER = [100_000, 250_000, 600_000, 1_400_000, 2_800_000, 4_300_000]

In [6]:
# ============================================================================
# 1. Pool construction, fixed val carve, nested prefixes
# ============================================================================
from data import create_example

def build_pool(seed=SEED):
    """All non-fundamental pairs, shuffled once. Fundamentals = both operands
    single-digit (a<10 and b<10) -> memorized lookup, excluded."""
    toks, masks = [], []
    for a in range(MAX_N):
        for b in range(MAX_N):
            if a < 10 and b < 10:
                continue
            t, m = create_example(a, b, MAX_LEN)
            toks.append(t)
            masks.append(m)
    toks  = np.asarray(toks,  dtype=np.int32)
    masks = np.asarray(masks, dtype=np.int32)
    perm = np.random.default_rng(seed).permutation(len(toks))
    return toks[perm], masks[perm]

def supervised_tokens(masks):
    return int(masks.sum())


def carve_val(pool_tok, pool_mask, val_sup_tokens=VAL_SUP_TOKENS):
    """Fixed val set taken from the front of the shuffled pool; the remainder
    is the training pool. Val is identical for every cell."""
    cum = np.cumsum(pool_mask.sum(axis=1))
    n_val = int(np.searchsorted(cum, val_sup_tokens) + 1)
    return (pool_tok[:n_val], pool_mask[:n_val],
            pool_tok[n_val:], pool_mask[n_val:])


def prefix_for_D(tr_tok, tr_mask, D_sup_tokens):
    """Nested prefix: fewest examples whose supervised tokens reach D."""
    cum = np.cumsum(tr_mask.sum(axis=1))
    n = min(int(np.searchsorted(cum, D_sup_tokens) + 1), len(tr_tok))
    return tr_tok[:n], tr_mask[:n]


def check_ladder(tr_mask, ladder=DATA_LADDER):
    """Verify every D is actually available; return the ceiling."""
    avail = supervised_tokens(tr_mask)
    print(f"train pool supervised tokens available: {avail:,}")
    for D in ladder:
        print(f"  D={D:>10,}  {'ok' if D <= avail else 'EXCEEDS POOL'}")
    if max(ladder) > avail:
        raise ValueError(
            f"largest D ({max(ladder):,}) exceeds pool ({avail:,}). "
            "Lower the top of DATA_LADDER.")
    return avail

In [7]:
# ============================================================================
# 2. Config / param counting
# ============================================================================
def make_cfg(base_cfg: Config, d_model, num_layers):
    """NOTE: if your Config does not derive heads from d_model, this keeps a
    fixed head count across the ladder. Adjust if you want key_dim pinned."""
    return base_cfg.replace(d_model=d_model, num_layers=num_layers)


def count_params(cfg: Config):
    w = Weights.init(cfg, jax.random.key(0))
    return int(sum(x.size for x in jax.tree_util.tree_leaves(w)))


In [8]:
# ============================================================================
# 3. Train one (N, D) cell — single epoch, matched cosine
# ============================================================================
import optax
from optax.losses import softmax_cross_entropy_with_integer_labels

def make_optimizer(peak_lr, total_steps, weight_decay=0.1):
    schedule = optax.warmup_cosine_decay_schedule(
        init_value=0.0,
        peak_value=peak_lr,
        # proportional warmup, capped at 50. The old max(50, ...) put 58% of a short
        # run into warmup, so small-D cells never trained at peak LR and flat-lined
        # at ~1.55 - which looked like a phase transition in the loss surface. It also
        # made decay_steps negative below 50 steps, killing those cells outright.
        warmup_steps=max(1, min(50, int(0.05 * total_steps))),
        decay_steps=total_steps,
        end_value=peak_lr * 0.1,
    )
    return optax.adamw(schedule, weight_decay=weight_decay)


def calc_val_loss(val_loader, weights):
    losses = [loss_fn(weights, x, m).item() for x, m in val_loader]
    return float(np.mean(losses))

def loss_fn(weights: jax.Array, token_ids: jax.Array, mask: jax.Array) -> jax.Array:
    logits = forward(token_ids[:, :-1], weights)
    targets = token_ids[:, 1:]
    # for addition, we need to build a mask because the model needs to only predict the last 3 digits
    # need mask code
    loss_mask = mask[:, 1:]
    loss = softmax_cross_entropy_with_integer_labels(logits, targets) # mean would also change since masking the outputs
    loss = jnp.sum(loss * loss_mask) / jnp.sum(loss_mask)
    return loss


# Single-seed noise exceeded the trend in the previous run: at D=600k the losses
# ran 0.62, 0.33, 1.54, 0.23, 0.13 as N increased - the 1.54 is a failed run, not a
# data point. Median over seeds is what makes the surface measurable.
# Cost scales linearly: set SEEDS = (0,) to go back to one run per cell.
SEEDS = (0, 1, 2)


def train_cell(base_cfg, d_model, num_layers, tr_tok, tr_mask,
               val_tok, val_mask, seeds=SEEDS):
    losses = [_train_one(base_cfg, d_model, num_layers, tr_tok, tr_mask,
                         val_tok, val_mask, sd) for sd in seeds]
    return float(np.median(losses)), float(max(losses) - min(losses)), len(losses)


def _train_one(base_cfg, d_model, num_layers, tr_tok, tr_mask,
               val_tok, val_mask, seed=SEED):
    cfg = make_cfg(base_cfg, d_model, num_layers)
    total_steps = len(tr_tok) // BATCH_SIZE      # single epoch
    if total_steps < 1:
        raise ValueError(f"D too small for one batch: {len(tr_tok)} examples")

    weights = Weights.init(cfg, jax.random.key(seed))
    opt = make_optimizer(PEAK_LR[(d_model, num_layers)], total_steps)
    opt_state = opt.init(weights)

    @jax.jit
    def train_step(x, mask, weights, opt_state):
        loss, grads = jax.value_and_grad(loss_fn)(weights, x, mask)
        updates, opt_state = opt.update(grads, opt_state, weights)
        weights = optax.apply_updates(weights, updates)
        return loss, weights, opt_state

    for x, m in Dataloader(tr_tok, tr_mask, BATCH_SIZE):
        _, weights, opt_state = train_step(x, m, weights, opt_state)

    return calc_val_loss(
        Dataloader(val_tok, val_mask, BATCH_SIZE, shuffle=False), weights)

In [9]:
# ============================================================================
# 4. Full-cross sweep (Part 1: measure the loss field, no compute constraint)
# ============================================================================
def run_sweep(base_cfg, seed=SEED):
    pool_tok, pool_mask = build_pool(seed)
    val_tok, val_mask, tr_tok, tr_mask = carve_val(pool_tok, pool_mask)

    print(f"pool: {len(pool_tok):,} ex, {supervised_tokens(pool_mask):,} sup-tok")
    print(f"val:  {len(val_tok):,} ex, {supervised_tokens(val_mask):,} sup-tok")
    D_ceiling = check_ladder(tr_mask)

    rows = []
    for (d_model, num_layers) in MODEL_LADDER:
        N = count_params(make_cfg(base_cfg, d_model, num_layers))
        for D in DATA_LADDER:
            sub_tok, sub_mask = prefix_for_D(tr_tok, tr_mask, D)
            D_actual = supervised_tokens(sub_mask)
            val_loss, spread, n_seeds = train_cell(
                base_cfg, d_model, num_layers, sub_tok, sub_mask, val_tok, val_mask)
            rows.append(dict(d_model=d_model, num_layers=num_layers, N=N,
                             D_target=D, D_actual=D_actual, val_loss=val_loss,
                             seed_spread=spread, n_seeds=n_seeds))
            print(f"N={N:>9,}  D={D_actual:>9,}  val_loss={val_loss:.4f}"
                  f" (+/-{spread:.3f} over {n_seeds} seeds)")
    return rows, D_ceiling

In [10]:
import json, time

# --- persist ---------------------------------------------------------------
# The Colab VM is ephemeral: anything written to /content disappears when the
# runtime recycles (this is how the previous rows.pkl was lost). Copy to Drive.
def save_rows(rows, D_ceiling, tag):
    def _jsonable(r):
        return {k: (v.tolist() if hasattr(v, 'tolist') else v) for k, v in r.items()}

    stamp = time.strftime('%Y%m%d-%H%M%S')
    path = f"rows_{tag}_{stamp}.json"
    payload = dict(rows=[_jsonable(r) for r in rows], D_ceiling=D_ceiling,
                   tag=tag, stamp=stamp)
    with open(path, "w") as f:
        json.dump(payload, f)
    print(f"saved {len(rows)} rows -> {path}")

    if IN_COLAB:
        from google.colab import drive
        import shutil, os
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive')
        dest_dir = '/content/drive/MyDrive/scaling-book'
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy(path, dest_dir)
        print(f"copied -> {dest_dir}/{path}")
    return path

rows, D_ceiling = run_sweep(base_cfg)
ROWS_PATH = save_rows(rows, D_ceiling, 'dense')



pool: 999,900 ex, 4,494,150 sup-tok
val:  22,231 ex, 100,003 sup-tok
train pool supervised tokens available: 4,394,147
  D=   100,000  ok
  D=   250,000  ok
  D=   600,000  ok
  D= 1,400,000  ok
  D= 2,800,000  ok
  D= 4,300,000  ok
N=  264,256  D=  100,003  val_loss=1.6321
N=  264,256  D=  250,001  val_loss=1.5530
N=  264,256  D=  600,002  val_loss=0.6175
N=  264,256  D=1,400,001  val_loss=0.1422
N=  264,256  D=2,800,002  val_loss=0.0449
N=  264,256  D=4,300,000  val_loss=0.0482
N=  667,296  D=  100,003  val_loss=1.5931
N=  667,296  D=  250,001  val_loss=1.5636
N=  667,296  D=  600,002  val_loss=0.3271
N=  667,296  D=1,400,001  val_loss=0.0546
N=  667,296  D=2,800,002  val_loss=0.0297
N=  667,296  D=4,300,000  val_loss=0.0246
N=  987,648  D=  100,003  val_loss=1.5779
N=  987,648  D=  250,001  val_loss=1.5541
N=  987,648  D=  600,002  val_loss=1.5373
N=  987,648  D=1,400,001  val_loss=0.2292
N=  987,648  D=2,800,002  val_loss=0.0319
N=  987,648  D=4,300,000  val_loss=0.0086
N=2,367,168

KeyboardInterrupt: 

In [ ]:
# ---------------------------------------------------------------------------
# Reload a previous sweep instead of retraining. Point at a rows_dense_*.json
# (local or on Drive) and every fit/plot cell below works unchanged.
# ---------------------------------------------------------------------------
# import json
# with open("rows_dense_YYYYmmdd-HHMMSS.json") as f:
#     _payload = json.load(f)
# rows, D_ceiling = _payload["rows"], _payload["D_ceiling"]
# print(f"loaded {len(rows)} rows, D_ceiling={D_ceiling:,}")


In [26]:
# ============================================================================
# 5. Fit L(N,D) = E + A/N^alpha + B/D^beta  (Huber on log-loss, multi-restart)
# ============================================================================
import contextlib

@contextlib.contextmanager
def x64_fit():
    """Run the fitter in float64.

    _predict_logL is a jax function and least_squares differentiates it by finite
    differences. Under jax's default float32 the nudge is below float32 resolution
    for the amplitude parameters, their computed gradient is exactly zero, and
    least_squares leaves them at their random starting values. The previous run of
    this notebook reported A=0.5517 and B=135 - those are restart #3's starting
    draws, not a fit. Scoped so training numerics are untouched.
    """
    old = jax.config.jax_enable_x64
    jax.config.update('jax_enable_x64', True)
    try:
        yield
    finally:
        jax.config.update('jax_enable_x64', old)


def _predict_logL(params, N, D):
    a, b, e, alpha, beta = params
    terms = jnp.stack([
        a - alpha * jnp.log(N),
        b - beta  * jnp.log(D),
        jnp.full_like(N, e),
    ], axis=0)
    return jax.scipy.special.logsumexp(terms, axis=0)


def fit_surface(rows, huber_delta=1e-3, n_restarts=30, seed=0):
    N = np.array([r['N'] for r in rows], dtype=np.float64)
    D = np.array([r['D_actual'] for r in rows], dtype=np.float64)
    logL = np.log(np.array([r['val_loss'] for r in rows], dtype=np.float64))
    Nj, Dj = jnp.asarray(N), jnp.asarray(D)

    def resid(p):
        return np.asarray(_predict_logL(p, Nj, Dj)) - logL

    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_restarts):
        p0 = np.array([rng.uniform(-2, 6),      # log A
                       rng.uniform(-2, 6),      # log B
                       rng.uniform(-2, 0),      # log E
                       rng.uniform(0.1, 0.9),   # alpha
                       rng.uniform(0.1, 0.9)])  # beta
        try:
            r = least_squares(resid, p0, loss='huber',
                              f_scale=huber_delta, max_nfev=10000)
        except Exception:
            continue
        if best is None or r.cost < best.cost:
            best = r

    if best is None:
        raise RuntimeError("all restarts failed")
    a, b, e, alpha, beta = best.x
    return dict(A=float(np.exp(a)), B=float(np.exp(b)), E=float(np.exp(e)),
                alpha=float(alpha), beta=float(beta),
                raw=best.x, cost=float(best.cost))


def bootstrap_fit(rows, n_boot=500, seed=0):
    rng = np.random.default_rng(seed)
    keys = ['A', 'B', 'E', 'alpha', 'beta']
    samples = {k: [] for k in keys}
    extra = {'p': [], 'q': []}
    n = len(rows)
    for _ in range(n_boot):
        sub = [rows[i] for i in rng.integers(0, n, n)]
        try:
            f = fit_surface(sub, n_restarts=8, seed=int(rng.integers(1e9)))
        except Exception:
            continue
        for k in keys:
            samples[k].append(f[k])
        s = f['alpha'] + f['beta']
        extra['p'].append(f['beta'] / s)
        extra['q'].append(f['alpha'] / s)

    def ci(arr):
        arr = np.array(arr)
        return dict(median=float(np.median(arr)),
                    lo=float(np.percentile(arr, 2.5)),
                    hi=float(np.percentile(arr, 97.5)))
    out = {k: ci(samples[k]) for k in keys}
    out['p'] = ci(extra['p'])
    out['q'] = ci(extra['q'])
    return out

In [28]:
# ============================================================================
# 6. Part 2: impose compute analytically. D = C/(6N), minimize over N.
# ============================================================================
def derive_frontier(fit, C_grid=None, D_ceiling=None):
    """Closed form from stationarity of A/N^alpha + B/(C/6N)^beta:
       N_opt = [ (alpha*A)/(beta*B) * (C/6)^beta ]^(1/(alpha+beta))
       so N_opt ~ C^p, D_opt ~ C^q with p=beta/(alpha+beta), q=alpha/(alpha+beta)."""
    A, B, alpha, beta = fit['A'], fit['B'], fit['alpha'], fit['beta']
    p = beta / (alpha + beta)
    q = alpha / (alpha + beta)

    if C_grid is None:
        C_grid = np.logspace(11, 18, 50)

    N_opt = ((alpha * A) / (beta * B) * (C_grid / 6.0) ** beta) ** (1.0 / (alpha + beta))
    D_opt = C_grid / (6.0 * N_opt)
    L_opt = fit['E'] + A / N_opt ** alpha + B / D_opt ** beta

    grounded = C_star = None
    if D_ceiling is not None:
        grounded = D_opt <= D_ceiling
        C_star = float(C_grid[grounded].max()) if grounded.any() else None

    return dict(C=C_grid, N_opt=N_opt, D_opt=D_opt, L_opt=L_opt,
                p=p, q=q, grounded=grounded, C_max_grounded=C_star)

In [29]:

# ============================================================================
# 7. Report
# ============================================================================
def report(fit, ci, fr):
    print("\n=== fitted surface L(N,D) = E + A/N^alpha + B/D^beta ===")
    for k in ['A', 'B', 'E', 'alpha', 'beta']:
        c = ci[k]
        print(f"  {k:>5} = {fit[k]:.4g}   95% CI [{c['lo']:.4g}, {c['hi']:.4g}]")
    print("\n=== compute-optimal frontier ===")
    print(f"  N_opt ~ C^{fr['p']:.4f}   95% CI [{ci['p']['lo']:.4f}, {ci['p']['hi']:.4f}]")
    print(f"  D_opt ~ C^{fr['q']:.4f}   95% CI [{ci['q']['lo']:.4f}, {ci['q']['hi']:.4f}]")
    if fr['C_max_grounded'] is not None:
        print(f"\n  data ceiling reached at C = {fr['C_max_grounded']:.3g} FLOPs")
        print("  beyond this the frontier extrapolates past the dataset "
              "(data-constrained regime; cf. Muennighoff et al. 2023)")

In [30]:
with x64_fit():
    fit = fit_surface(rows)
    ci  = bootstrap_fit(rows, n_boot=500)
fr  = derive_frontier(fit, D_ceiling=D_ceiling)
report(fit, ci, fr)


=== fitted surface L(N,D) = E + A/N^alpha + B/D^beta ===
      A = 0.5517   95% CI [0.1493, 262.7]
      B = 135   95% CI [0.1954, 709.4]
      E = 0.1144   95% CI [0.0006283, 0.4635]
  alpha = 0.477   95% CI [0.2004, 6.996]
   beta = 0.4381   95% CI [0.1591, 5.258]

=== compute-optimal frontier ===
  N_opt ~ C^0.4788   95% CI [0.0635, 0.8921]
  D_opt ~ C^0.5212   95% CI [0.1079, 0.9365]


In [31]:
def diagnose_bootstrap(rows, n_boot=500, seed=0):
    rng = np.random.default_rng(seed)
    a_s, b_s, e_s = [], [], []
    n = len(rows)
    for _ in range(n_boot):
        sub = [rows[i] for i in rng.integers(0, n, n)]
        try:
            f = fit_surface(sub, n_restarts=8, seed=int(rng.integers(1e9)))
        except Exception:
            continue
        a_s.append(f['alpha']); b_s.append(f['beta']); e_s.append(f['E'])
    a_s, b_s, e_s = map(np.array, (a_s, b_s, e_s))
    for name, arr in [('alpha', a_s), ('beta', b_s)]:
        print(f"{name}: median {np.median(arr):.3f}  "
              f"IQR [{np.percentile(arr,25):.3f}, {np.percentile(arr,75):.3f}]  "
              f"frac > 1.0: {(arr > 1.0).mean():.3f}")
    print(f"corr(alpha, log E) = {np.corrcoef(a_s, np.log(e_s))[0,1]:.3f}")
    return a_s, b_s, e_s

with x64_fit():
    diagnose_bootstrap(rows)

alpha: median 0.678  IQR [0.502, 0.870]  frac > 1.0: 0.208
beta: median 0.509  IQR [0.368, 0.686]  frac > 1.0: 0.066
corr(alpha, log E) = 0.069


(array([ 0.39165282,  0.49073114,  0.84525637,  0.2798987 ,  0.66969239,
         0.74236049,  4.80755757,  0.50784443,  0.82499613,  5.70514305,
         0.73954958,  2.72119493,  0.69684261,  0.32847263,  0.4014916 ,
         0.37812599,  0.77776391,  0.89750646,  0.74628067,  0.71326075,
         0.57259626,  0.87714356,  0.7992693 ,  0.76564478,  0.55822344,
         0.22494558,  0.53950164,  0.16333883,  0.35092803,  0.35372145,
         0.61249263,  0.38945382,  0.76889691,  0.55542824,  0.52227936,
         0.41640593,  0.75849392,  0.69827374,  0.88315844,  0.5522495 ,
         0.47742872,  7.15426099,  0.52698735,  0.54508528,  0.75327946,
         3.14668503,  5.38486106,  0.28704215,  0.81642963,  0.6848769 ,
         4.51306199,  0.37797413,  0.72264052,  0.59203074,  5.42771508,
         0.69833226,  0.40311345,  0.80396099,  0.89861665,  0.87191872,
         0.62545308,  0.6326403 ,  0.4226541 ,  0.8897161 ,  0.68140421,
         5.45028753,  0.27846932,  0.6384528 ,  0.7